In [14]:
%matplotlib inline

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Instacart Market Basket Analysis

## Part 1: Data Cleaning and Exploratory Data Analysis

This notebook covers loading the six raw Instacart tables, validating and cleaning them, and the exploratory analysis that motivates every downstream feature engineering and modelling decision in this project.

The two sections are kept deliberately separate. Cleaning establishes that the data is trustworthy: correct dtypes, no impossible values, no duplicate keys, and a documented explanation for every null. EDA then asks what the data actually says: how users behave, how products get reordered, what the class balance looks like, and where the recall ceiling sits before any model is trained. Every EDA finding below is connected explicitly to a decision made later in feature engineering or modelling, rather than included for its own sake.

## Phase 1: Foundation, Data Loading

Goal: load the six raw CSV files, inspect schema and data quality, validate value ranges before touching dtypes, downcast only after validation passes, and save to parquet.

In [16]:
RAW_DIR = Path("../data/raw")

orders = pd.read_csv(RAW_DIR / "orders.csv")
order_products_prior = pd.read_csv(RAW_DIR / "order_products__prior.csv")
order_products_train = pd.read_csv(RAW_DIR / "order_products__train.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
aisles = pd.read_csv(RAW_DIR / "aisles.csv")
departments = pd.read_csv(RAW_DIR / "departments.csv")

tables = {
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
    "products": products,
    "aisles": aisles,
    "departments": departments,
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (3421083, 7)
order_products_prior: (32434489, 4)
order_products_train: (1384617, 4)
products: (49688, 4)
aisles: (134, 2)
departments: (21, 2)


### Step 2: Inspect schema, dtypes, and nulls

In [17]:
for name, df in tables.items():
    print(f"\n{name}")
    print(df.dtypes)
    nulls = df.isnull().sum()
    print(nulls[nulls > 0])


orders
order_id                    int64
user_id                     int64
eval_set                   object
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object
days_since_prior_order    206209
dtype: int64

order_products_prior
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object
Series([], dtype: int64)

order_products_train
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object
Series([], dtype: int64)

products
product_id        int64
product_name     object
aisle_id          int64
department_id     int64
dtype: object
Series([], dtype: int64)

aisles
aisle_id     int64
aisle       object
dtype: object
Series([], dtype: int64)

departments
department_id     int64
department       object
dtype: object
Series([], dtype: int64)


Check whether `days_since_prior_order` nulls are structural, meaning they only occur on each user's first order, rather than being a data quality problem.

In [18]:
first_orders = orders[orders["order_number"] == 1]
null_dspo_on_first = first_orders["days_since_prior_order"].isnull().mean()
print(f"Fraction of first orders with null days_since_prior_order: {null_dspo_on_first:.4f}")

non_first_orders = orders[orders["order_number"] > 1]
null_dspo_on_non_first = non_first_orders["days_since_prior_order"].isnull().mean()
print(f"Fraction of non-first orders with null days_since_prior_order: {null_dspo_on_non_first:.4f}")

Fraction of first orders with null days_since_prior_order: 1.0000
Fraction of non-first orders with null days_since_prior_order: 0.0000


If the first fraction is near 1.0 and the second is near 0.0, the null pattern is structural and does not need imputation, just documentation.

### Step 3: Validate value ranges before any dtype change

Before downcasting, check that every integer column actually fits in int32, and check for values that should never occur, negative IDs, negative quantities, zero where zero is impossible.

In [19]:
def validate_int_column(df, col, allow_zero=True, allow_negative=False):
    min_val = df[col].min()
    max_val = df[col].max()
    int32_min, int32_max = np.iinfo(np.int32).min, np.iinfo(np.int32).max
    fits = (min_val >= int32_min) and (max_val <= int32_max)
    print(f"{col}: min={min_val}, max={max_val}, fits int32={fits}")
    if not allow_negative and min_val < 0:
        print(f"  WARNING: {col} has negative values, this should not happen")
    if not allow_zero and (df[col] == 0).any():
        print(f"  WARNING: {col} has zero values, check if expected")

In [20]:
print("orders.csv")
validate_int_column(orders, "order_id")
validate_int_column(orders, "user_id")
validate_int_column(orders, "order_number")
validate_int_column(orders, "order_dow")
validate_int_column(orders, "order_hour_of_day")

orders.csv
order_id: min=1, max=3421083, fits int32=True
user_id: min=1, max=206209, fits int32=True
order_number: min=1, max=100, fits int32=True
order_dow: min=0, max=6, fits int32=True
order_hour_of_day: min=0, max=23, fits int32=True


In [21]:
print("order_products_prior.csv")
validate_int_column(order_products_prior, "order_id")
validate_int_column(order_products_prior, "product_id")
validate_int_column(order_products_prior, "add_to_cart_order", allow_zero=False)
validate_int_column(order_products_prior, "reordered")

order_products_prior.csv
order_id: min=2, max=3421083, fits int32=True
product_id: min=1, max=49688, fits int32=True
add_to_cart_order: min=1, max=145, fits int32=True
reordered: min=0, max=1, fits int32=True


In [22]:
print("order_products_train.csv")
validate_int_column(order_products_train, "order_id")
validate_int_column(order_products_train, "product_id")
validate_int_column(order_products_train, "add_to_cart_order", allow_zero=False)
validate_int_column(order_products_train, "reordered")

order_products_train.csv
order_id: min=1, max=3421070, fits int32=True
product_id: min=1, max=49688, fits int32=True
add_to_cart_order: min=1, max=80, fits int32=True
reordered: min=0, max=1, fits int32=True


In [23]:
print("products.csv")
validate_int_column(products, "product_id")
validate_int_column(products, "aisle_id")
validate_int_column(products, "department_id")

products.csv
product_id: min=1, max=49688, fits int32=True
aisle_id: min=1, max=134, fits int32=True
department_id: min=1, max=21, fits int32=True


Check the float column separately, since it can be negative in principle (it isn't expected to be, but check) and needs a different validation than an ID column.

In [25]:
dspo = orders["days_since_prior_order"]
print(f"days_since_prior_order: min={dspo.min()}, max={dspo.max()}")
print(f"Negative values: {(dspo < 0).sum()}")
print(f"Values above 30 (the stated cap): {(dspo > 30).sum()}")

days_since_prior_order: min=0.0, max=30.0
Negative values: 0
Values above 30 (the stated cap): 0


### Step 4: Check for duplicate rows

Confirm there are no duplicate (user, order) pairs, and no duplicate (order_id, product_id) pairs within the order_products tables.

In [26]:
dup_orders = orders.duplicated(subset=["order_id"]).sum()
print(f"Duplicate order_id in orders: {dup_orders}")

dup_op_prior = order_products_prior.duplicated(subset=["order_id", "product_id"]).sum()
print(f"Duplicate (order_id, product_id) in order_products_prior: {dup_op_prior}")

dup_op_train = order_products_train.duplicated(subset=["order_id", "product_id"]).sum()
print(f"Duplicate (order_id, product_id) in order_products_train: {dup_op_train}")

Duplicate order_id in orders: 0
Duplicate (order_id, product_id) in order_products_prior: 0
Duplicate (order_id, product_id) in order_products_train: 0


In [27]:
order_ids_in_orders = set(orders["order_id"])
order_ids_in_prior = set(order_products_prior["order_id"])
order_ids_in_train = set(order_products_train["order_id"])

print(f"Prior order_ids not found in orders: {len(order_ids_in_prior - order_ids_in_orders)}")
print(f"Train order_ids not found in orders: {len(order_ids_in_train - order_ids_in_orders)}")
print(f"Order_ids appearing in both prior and train: {len(order_ids_in_prior & order_ids_in_train)}")

Prior order_ids not found in orders: 0
Train order_ids not found in orders: 0
Order_ids appearing in both prior and train: 0


### Cleaning Summary

All six tables pass validation. Schema and dtypes match expectations, with no unexpected nulls outside `days_since_prior_order`, whose null pattern is exactly structural: null if and only if `order_number == 1`. Value ranges fall within documented bounds (`days_since_prior_order` in [0, 30], no negative IDs or quantities). No duplicate `order_id` in orders, no duplicate `(order_id, product_id)` pairs within either order_products table, no orphaned `order_id` values in either order_products table relative to orders, and no `order_id` appears in both `order_products_prior` and `order_products_train`. The data is ready for downcasting and the EDA section below.